In [1]:
import pandas as pd
import numpy as np
from dash import Dash, html, dcc, Input, Output, dash_table
import dash_bootstrap_components as dbc
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [2]:
farm_files = {
    "Farm 66": "Data/Data_Folder/updated-animal-data-farm-id-66.csv",
    "Farm 68": "Data/Data_Folder/updated-animal-data-farm-id-68.csv",
    "Farm 70": "Data/Data_Folder/updated-animal-data-farm-id-70.csv",
    "Farm 73": "Data/Data_Folder/updated-animal-data-farm-id-73.csv",
    "Farm 74": "Data/Data_Folder/updated-animal-data-farm-id-74.csv",
    "Farm 75": "Data/Data_Folder/updated-animal-data-farm-id-75.csv",
    "New 66": "Data/Extra Stuff/new-animal-data-farm-id-66-2025-10-27-to-2025-11-19.csv"
}

def load_farm_data(farm_name):
   
    df = pd.read_csv(farm_files[farm_name])

    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')

    df['airTemp'] = pd.to_numeric(df.get('airTemp', pd.Series()), errors='coerce')
    df['humidity'] = pd.to_numeric(df.get('humidity', pd.Series()), errors='coerce')

    df['THI'] = (0.8 * df['airTemp'] + 0.01 * df['humidity'] * (df['airTemp'] - 14.4) + 46.4)

    df['THI'] = df['THI'].round(2)

    return df

In [3]:
def countUniqueIDs(selected_farm):
    df = load_farm_data(selected_farm)
    unique_ids = df['id'].unique()
    return len(unique_ids)

In [4]:

from datetime import timedelta

def create_temp_plot(data, color1='#F4D03F', color2='skyblue', color3='lightgreen'):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=data['datetime'], y=data['temp'], mode='lines',
        name='Calf Temp (°C)', line=dict(color=color1, width=2)
    ))
    fig.add_trace(go.Scatter(
        x=data['datetime'], y=data['airTemp'], mode='lines',
        name='Outside Air Temp (°C)', line=dict(color=color2, width=2, dash='dot')
    ))
    fig.add_trace(go.Scatter(
        x=data['datetime'], y=data['THI'], mode='lines', yaxis='y2',
        name='THI', line=dict(color=color3, width=2, dash='dash')
    ))

    data['month'] = data['datetime'].dt.month

    # --- Summer months ---
    summer_mask = data['month'].isin([5, 6, 7, 8])

    # Heat stress (high THI + high temp)
    heat_stress = data[(summer_mask) & (data['THI'] > 72) & (data['temp'] > 40)]

    # Non-heat fever (summer, normal THI but high temp)
    non_heat_summer = data[(summer_mask) & (data['THI'] <= 72) & (data['temp'] > 40)]

    # Recurrent fever (non-summer)
    non_summer = data[~summer_mask].copy()
    non_summer['date'] = non_summer['datetime'].dt.date
    recurrent_days = (
        non_summer[non_summer['temp'] > 40]
        .groupby('date').filter(lambda x: len(x) > 1)
    )

    # Heat stress fever markers
    fig.add_trace(go.Scatter(
        x=heat_stress['datetime'], y=heat_stress['temp'],
        mode='markers', name='🔥 Heat Stress Fever',
        marker=dict(color='red', size=8, symbol='circle'),
        hovertext=[f"Temp={t:.1f}°C, THI={h:.1f}" for t, h in zip(heat_stress['temp'], heat_stress['THI'])]
    ))

    # Non-heat fever markers
    fig.add_trace(go.Scatter(
        x=non_heat_summer['datetime'], y=non_heat_summer['temp'],
        mode='markers', name='⚠️ Non-Heat Fever (Summer)',
        marker=dict(color='green', size=8, symbol='square'),
        hovertext=[f"Temp={t:.1f}°C, THI={h:.1f}" for t, h in zip(non_heat_summer['temp'], non_heat_summer['THI'])]
    ))

    # Recurrent fever markers
    fig.add_trace(go.Scatter(
        x=recurrent_days['datetime'], y=recurrent_days['temp'],
        mode='markers', name='🩺 Recurrent Fever (Non-Heat)',
        marker=dict(color='blue', size=8, symbol='diamond'),
        hovertext=[f"Temp={t:.1f}°C" for t in recurrent_days['temp']]
    ))

    # --- Event markers ---
    non_null_events = data[data['event'].notnull()]
    if not non_null_events.empty:
        fig.add_trace(go.Scatter(
            x=non_null_events['datetime'],
            y=non_null_events['temp'],
            mode='markers+text',
            name='Events',
            marker=dict(symbol='x', size=12, color='red', line=dict(width=2, color='white')),
            text=non_null_events['event'],
            textposition='top center',
            textfont=dict(color='white', size=10),
            hovertemplate="<b>Event:</b> %{text}<br><b>Date:</b> %{x|%d %b %Y %H:%M}<br><b>Temp:</b> %{y:.1f}°C"
        ))

    # --- Add preclinical window as shaded vertical rectangles (vrect) ---
    for idx, row in non_null_events.iterrows():
        event_datetime = row['datetime']
        start = (event_datetime - timedelta(days=5)).replace(hour=0, minute=0, second=0, microsecond=0)
        end = event_datetime  # exact time of event

        fig.add_vrect(
            x0=start, x1=end,
            fillcolor="orange", opacity=0.15, layer="below", line_width=0,
            annotation_text="Preclinical Window",
            annotation_position="top left",
            annotation_font=dict(color="orange", size=10)
        )

    # --- Layout ---
    fig.update_layout(
        title=f'🐄 Calf ID {data["id"].iloc[0]} — Temperature, Air Temp & THI',
        xaxis_title='Date & Time',
        yaxis_title='Temperature (°C)',
        legend_title='Metrics',
        template='plotly_dark',
        hovermode='x unified',
        yaxis2=dict(title='THI', overlaying='y', side='right', showgrid=False),
        title_font=dict(size=20, family='Arial', color='white')
    )

    return fig



In [ ]:
def create_rumination_plot(df):
    """
    Create an interactive rumination plot for a specific calf.

    Highlights:
    - Ideal rumination range (blue band)
    - Preclinical windows (orange shade)
    - Event markers (red X)
    - Low rumination points inside clinical window (red dots)
    """

    df['date'] = pd.to_datetime(df['date'])

    daily_rumination = df.groupby('date').agg({
        'rumination': 'sum',
        'ageInDays': 'mean'
    }).reset_index()

    daily_rumination['rumination_hours'] = daily_rumination['rumination'] * 0.25

    def ideal_rumination_range(age):
        if pd.isna(age):
            return (None, None)
        elif age < 14:
            return (0, 2)
        elif age < 30:
            return (2, 4)
        elif age < 60:
            return (4, 8)
        else:
            return (6, 12)

    daily_rumination['ideal_min'], daily_rumination['ideal_max'] = zip(
        *daily_rumination['ageInDays'].apply(ideal_rumination_range)
    )


    non_null_events = df[df['event'].notnull()].copy()
    non_null_events['date'] = pd.to_datetime(non_null_events['date'])


    preclinical_days = []
    for d in non_null_events['date']:
        window = pd.date_range(d - pd.Timedelta(days=5), d - pd.Timedelta(days=1))
        preclinical_days.extend(window)
        fig_vrect_color = "orange"
    
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=pd.concat([daily_rumination['date'], daily_rumination['date'][::-1]]),
        y=pd.concat([daily_rumination['ideal_max'], daily_rumination['ideal_min'][::-1]]),
        fill='toself',
        fillcolor='rgba(135, 206, 250, 0.2)',
        line=dict(color='rgba(255,255,255,0)'),
        name='Ideal Range (hrs/day)',
        hoverinfo='skip'
    ))

    fig.add_trace(go.Scatter(
        x=daily_rumination['date'],
        y=daily_rumination['rumination_hours'],
        mode='lines+markers',
        name='Actual Rumination (hrs/day)',
        line=dict(color='lightgreen', width=3, shape='spline'),
        marker=dict(size=8, color='lightgreen'),
        hovertemplate="<b>Date:</b> %{x|%d %b %Y}<br>"
                      "<b>Rumination:</b> %{y:.2f} hrs<br>"
                      "<b>Age:</b> %{customdata:.0f} days",
        customdata=daily_rumination['ageInDays']
    ))

    for d in non_null_events['date']:
        start = d - pd.Timedelta(days=5)
        end = d - pd.Timedelta(days=1)
        fig.add_vrect(
            x0=start, x1=end,
            fillcolor="orange", opacity=0.15, layer="below", line_width=0,
            annotation_text="Preclinical Window",
            annotation_position="top left",
            annotation_font=dict(color="orange", size=10)
        )

    # --- Identify low rumination in clinical window ---
    daily_rumination['in_window'] = daily_rumination['date'].isin(preclinical_days)
    daily_rumination['below_range'] = daily_rumination['rumination_hours'] < daily_rumination['ideal_min']

    low_points = daily_rumination[daily_rumination['in_window'] & daily_rumination['below_range']]

    # --- Add low rumination markers (inside clinical window) ---
    if not low_points.empty:
        fig.add_trace(go.Scatter(
            x=low_points['date'],
            y=low_points['rumination_hours'],
            mode='markers',
            name='Low Rumination (Preclinical)',
            marker=dict(size=10, color='red', symbol='circle', line=dict(width=1, color='white')),
            hovertemplate="<b>Date:</b> %{x|%d %b %Y}<br>"
                          "<b>Rumination:</b> %{y:.2f} hrs<br>"
                          "<b>Status:</b> Below Ideal (Preclinical)"
        ))

    # --- Event markers (red X) ---
    fig.add_trace(go.Scatter(
        x=non_null_events['date'],
        y=[
            daily_rumination.set_index('date').loc[d, 'rumination_hours']
            if d in daily_rumination['date'].values else None
            for d in non_null_events['date']
        ],
        mode='markers+text',
        name='Events',
        marker=dict(symbol='x', size=12, color='red', line=dict(width=2, color='white')),
        text=non_null_events['event'],
        textposition='top center',
        textfont=dict(color='white', size=12),
        hovertemplate="<b>Event:</b> %{text}<br><b>Date:</b> %{x|%d %b %Y}"
    ))

    fig.update_layout(
        title=f"🐄 Daily Rumination Pattern — Calf ID {df['id'].iloc[0]}",
        xaxis_title="Date",
        yaxis_title="Rumination (hours per day)",
        template="plotly_dark",
        hovermode="x unified",
        legend=dict(title="Legend", orientation="h", y=-0.25, x=0.3),
        title_font=dict(size=22, family="Arial", color="white"),
        xaxis=dict(showgrid=False),
        yaxis=dict(showgrid=True, gridcolor="gray"),
        margin=dict(l=60, r=30, t=80, b=80)
    )

    return fig


In [ ]:
def calculate_hourly_rolling_average(df, window_hours=24):
    """Calculate rolling average for activity data"""
    df = df.copy()
    df = df.sort_values('datetime')
    
    # Aggregate to hourly averages first (from 15-min data)
    df['hour'] = df['datetime'].dt.floor('H')
    hourly_df = df.groupby('hour').agg({
        'accel': ['mean', 'max', 'min'],
        'datetime': 'first'
    }).reset_index()
    
    # Flatten column names
    hourly_df.columns = ['hour', 'accel_avg', 'accel_max', 'accel_min', 'datetime']
    
    # Calculate rolling average
    hourly_df[f'rolling_{window_hours}h'] = hourly_df['accel_avg'].rolling(
        window=window_hours, 
        min_periods=1
    ).mean()
    
    return hourly_df


def get_daily_activity_summary(df):
    """Get daily activity summary"""
    df = df.copy()
    df['date'] = df['datetime'].dt.date
    
    daily_summary = df.groupby('date').agg({
        'accel': ['mean', 'sum', 'max', 'min', 'std']
    }).reset_index()
    
    daily_summary.columns = ['date', 'avg', 'total', 'max', 'min', 'std']
    return daily_summary


def create_activity_plot(df):
    """
    Create enhanced activity plot with two subplots:
    1. Daily overview with bars and trend line
    2. Hourly detail with 24-hour rolling average
    
    Features:
    - Daily average bars (color-coded by threshold)
    - Hourly raw data with 24h rolling average
    - Event markers on both plots
    - Preclinical windows (5 days before events)
    - 0.1 threshold line
    """
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import pandas as pd
    from datetime import timedelta
    
    # Calculate summaries
    daily_activity = get_daily_activity_summary(df)
    hourly_activity = calculate_hourly_rolling_average(df, window_hours=24)
    
    # Ensure dates are datetime
    daily_activity['date'] = pd.to_datetime(daily_activity['date'])
    
    # Create subplots
    fig = make_subplots(
        rows=2, cols=1,
        row_heights=[0.5, 0.5],
        subplot_titles=(
            f"Daily Activity Overview - Calf ID {df['id'].iloc[0]}", 
            "⏱ Hourly Activity with 24-Hour Rolling Average"
        ),
        vertical_spacing=0.12
    )
    
    # ==================== SUBPLOT 1: DAILY OVERVIEW ====================
    
    # Bar chart for daily averages (color-coded)
    bar_colors = ['#ef4444' if avg < 0.15 else '#3b82f6' for avg in daily_activity['avg']]
    
    fig.add_trace(go.Bar(
        x=daily_activity['date'],
        y=daily_activity['avg'],
        name='Daily Average',
        marker_color=bar_colors,
        hovertemplate='<b>Date:</b> %{x|%d %b %Y}<br><b>Avg Activity:</b> %{y:.3f}<extra></extra>',
        showlegend=True
    ), row=1, col=1)
    
    # Trend line
    fig.add_trace(go.Scatter(
        x=daily_activity['date'],
        y=daily_activity['avg'],
        mode='lines+markers',
        name='Daily Trend',
        line=dict(color='lightgreen', width=2),
        marker=dict(size=6, color='lightgreen'),
        hovertemplate='<b>Date:</b> %{x|%d %b %Y}<br><b>Trend:</b> %{y:.3f}<extra></extra>'
    ), row=1, col=1)
    
    # ==================== SUBPLOT 2: HOURLY DETAIL ====================
    
    # Raw hourly data (lighter)
    fig.add_trace(go.Scatter(
        x=hourly_activity['datetime'],
        y=hourly_activity['accel_avg'],
        mode='lines',
        name='Hourly Average',
        line=dict(color='#cbd5e1', width=1),
        hovertemplate='<b>Time:</b> %{x}<br><b>Hourly Avg:</b> %{y:.3f}<extra></extra>',
        showlegend=True
    ), row=2, col=1)
    
    # 24-hour rolling average (prominent)
    fig.add_trace(go.Scatter(
        x=hourly_activity['datetime'],
        y=hourly_activity['rolling_24h'],
        mode='lines',
        name='24h Rolling Average',
        line=dict(color='lightgreen', width=3),
        hovertemplate='<b>Time:</b> %{x}<br><b>24h Rolling Avg:</b> %{y:.3f}<extra></extra>',
        showlegend=True
    ), row=2, col=1)
    
    # ==================== THRESHOLD LINES ====================
    
    # Add 0.1 threshold to both plots
    fig.add_hline(
        y=0.1,
        line=dict(color='#ef4444', width=2, dash='dash'),
        row=1, col=1,
        annotation=dict(
            text="Low Activity Threshold (0.1)",
            font=dict(size=10, color='#ef4444'),
            xanchor='right',
            x=1
        )
    )
    
    fig.add_hline(
        y=0.1,
        line=dict(color='#ef4444', width=2, dash='dash'),
        row=2, col=1,
        annotation=dict(
            text="Low Activity Threshold",
            font=dict(size=10, color='#ef4444'),
            xanchor='right',
            x=1
        )
    )
    
    # ==================== EVENT MARKERS & PRECLINICAL WINDOWS ====================
    
    non_null_events = df[df['event'].notnull()].copy()
    
    if not non_null_events.empty:
        # Get unique event dates
        non_null_events['event_date'] = pd.to_datetime(non_null_events['date'])
        event_dates = non_null_events['event_date'].unique()
        
        # Add preclinical windows (5 days before each event) to BOTH plots
        for event_date in event_dates:
            start = pd.to_datetime(event_date) - timedelta(days=5)
            end = pd.to_datetime(event_date)
            
            # Daily plot window
            fig.add_vrect(
                x0=start, x1=end,
                fillcolor="orange", opacity=0.15, 
                layer="below", line_width=0,
                annotation_text="Preclinical",
                annotation_position="top left",
                annotation_font=dict(color="orange", size=9),
                row=1, col=1
            )
            
            # Hourly plot window
            fig.add_vrect(
                x0=start, x1=end,
                fillcolor="orange", opacity=0.15, 
                layer="below", line_width=0,
                row=2, col=1
            )
        
        # Event markers on DAILY plot
        event_daily = daily_activity[daily_activity['date'].isin(event_dates)]
        event_info = non_null_events.groupby('event_date')['event'].first().reset_index()
        event_info.columns = ['date', 'event']
        event_plot_data = event_daily.merge(event_info, on='date', how='inner')
        
        if not event_plot_data.empty:
            fig.add_trace(go.Scatter(
                x=event_plot_data['date'],
                y=event_plot_data['avg'],
                mode='markers+text',
                name='Events',
                marker=dict(symbol='x', size=14, color='red', line=dict(width=2, color='white')),
                text=event_plot_data['event'],
                textposition='top center',
                textfont=dict(color='white', size=11),
                hovertemplate='<b>Event:</b> %{text}<br><b>Date:</b> %{x|%d %b %Y}<extra></extra>',
                showlegend=False
            ), row=1, col=1)
        
        # Event markers on HOURLY plot (placed on rolling average line)
        non_null_events['event_datetime'] = pd.to_datetime(non_null_events['datetime'])
        
        for _, event_row in non_null_events.iterrows():
            event_dt = event_row['event_datetime']
            # Find closest hourly point
            closest_idx = (hourly_activity['datetime'] - event_dt).abs().idxmin()
            event_y_value = hourly_activity.loc[closest_idx, 'rolling_24h']
            
            fig.add_trace(go.Scatter(
                x=[event_dt],
                y=[event_y_value],
                mode='markers+text',
                marker=dict(symbol='x', size=14, color='red', line=dict(width=2, color='white')),
                text=[event_row['event']],
                textposition='top center',
                textfont=dict(color='white', size=11),
                hovertemplate=f"<b>Event:</b> {event_row['event']}<br><b>Time:</b> {event_dt}<extra></extra>",
                showlegend=False
            ), row=2, col=1)
    
    # ==================== LAYOUT CONFIGURATION ====================
    
    fig.update_xaxes(title_text="Date", row=1, col=1, showgrid=False)
    fig.update_xaxes(title_text="Date & Time", row=2, col=1, showgrid=False)
    
    fig.update_yaxes(title_text="Activity (avg accel)", row=1, col=1, showgrid=True, gridcolor='gray')
    fig.update_yaxes(title_text="Activity (accel)", row=2, col=1, showgrid=True, gridcolor='gray')
    
    fig.update_layout(
        template="plotly_dark",
        hovermode="x unified",
        height=900,
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.15,
            xanchor="center",
            x=0.5,
            bgcolor="rgba(0,0,0,0.5)"
        ),
        margin=dict(l=60, r=30, t=100, b=100)
    )
    
    return fig

In [ ]:
def eventTable(selected_farm, selected_id):
    df = load_farm_data(selected_farm)
    df = df[df['id'] == selected_id]

    events = df[df['event'].notnull()].copy()

    events['datetime'] = pd.to_datetime(events['datetime'])


    events['date'] = events['datetime'].dt.strftime('%#d %B, %Y')  
    events['time'] = events['datetime'].dt.strftime('%H:%M:%S')   

    events['date'] = events['date'].astype(str)
    events['time'] = events['time'].astype(str)

    table_data = events[['id', 'event', 'date', 'time', 'temp', 'THI']].rename(
        columns={'temp': 'body_Temperature'}
    )

    return table_data


C:\Users\rashe\AppData\Local\Temp\ipykernel_2008\3921853781.py:13: DtypeWarning:

Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\rashe\AppData\Local\Temp\ipykernel_2008\3921853781.py:13: DtypeWarning:

Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\rashe\AppData\Local\Temp\ipykernel_2008\3921853781.py:13: DtypeWarning:

Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\rashe\AppData\Local\Temp\ipykernel_2008\3921853781.py:13: DtypeWarning:

Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\rashe\AppData\Local\Temp\ipykernel_2008\3921853781.py:13: DtypeWarning:

Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\rashe\AppData\Local\Temp\ipykernel_2008\3921853781.py:13: DtypeWarning:

Columns (1) have mixed types. Specify dtype option on import or set low_m